# Lesson 4.4: Feature Scaling

## Masla
10 overs ke baad:
- **runs_10**: aam tor pe 55 se 110
- **wickets_10**: aam tor pe 0 se 5

Computer sirf numbers dekhta hai. Usay "20 runs ka farq", "2 wickets ke farq" se **10 guna bara** lagta hai, jabke cricket mein 2 wickets match badal deti hain.
Bari range wala column chupke se **haavi** ho jata hai.

## Hal: dono ko ek hi zubaan mein badlo
Har value ke liye poocho: *"Ye average se kitni upar ya neeche hai, std ke hisaab se?"*

```
scaled value = (value - average) / std
```

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

---
## Part 1: Haath se scaling

4 chhote matches. Runs ka average **80**, std **10**. Wickets ka average **2**, std **1**.

In [2]:
small = pd.DataFrame({
    "runs_10":    [70, 90, 70, 90],
    "wickets_10": [1,  1,  3,  3],
})
small

,runs_10,wickets_10
0,70,1
1,90,1
2,70,3
3,90,3


In [3]:
# Har column ka average aur std
# ddof=0 wala std hi scikit-learn use karta hai
mean = small.mean()
std = small.std(ddof=0)

print("Average:")
print(mean)
print("Std:")
print(std)

Average:
runs_10       80.0
wickets_10     2.0
dtype: float64
Std:
runs_10       10.0
wickets_10     1.0
dtype: float64


In [4]:
# Formula har value pe ek saath (vectorisation)
scaled = (small - mean) / std
scaled

,runs_10,wickets_10
0,-1.0,-1.0
1,1.0,-1.0
2,-1.0,1.0
3,1.0,1.0


### Natija
Pehle runs ka farq **20** tha aur wickets ka **2**. Ab dono ka farq **2** hai (-1 se +1). Dono **ek hi zubaan** bol rahe hain.

| Scaled value | Matlab |
|---|---|
| 0 | bilkul average |
| +1 | average se ek std upar |
| -1 | average se ek std neeche |
| +2 | average se bohat upar |

---
## Part 2: Yahi kaam StandardScaler se

| Method | Kaam |
|---|---|
| `fit()` | mean aur std **seekh ke yaad rakho** |
| `transform()` | formula `(value - mean) / std` **lagao** |

In [5]:
scaler = StandardScaler()
scaler.fit(small)

print("Mean:", scaler.mean_)   # [80, 2]
print("Std: ", scaler.scale_)  # [10, 1]

Mean: [80.  2.]
Std:  [10.  1.]


In [6]:
# Part 1 wali same table (bas NumPy array ki shakal mein)
scaler.transform(small)

array([[-1., -1.],
       [ 1., -1.],
       [-1.,  1.],
       [ 1.,  1.]])

### Naye matches ko scale karna
Scaler ne mean aur std **yaad** kar liye hain, to naye matches pe **sirf transform** karte hain, dobara fit nahi.

In [7]:
new_matches = pd.DataFrame({
    "runs_10":    [80, 100],
    "wickets_10": [2,   4],
})

scaler.transform(new_matches)

array([[0., 0.],
       [2., 2.]])

### Natija
- **80/2 -> (0, 0)**: bilkul average match
- **100/4 -> (2, 2)**: runs mein bohat tez start, lekin wickets bhi bohat zyada gir chuki hain

**Yaad rakhein:** plus/minus "achha/bura" nahi batata, sirf "average se upar/neeche" batata hai.
Wickets mein +2 batting team ke liye bura hai, runs mein +2 achha.

---
## Part 3: `fit` sirf training data pe kyun?

### Experiment 1: Akela naya match (jaise FastAPI form se aata hai)

In [8]:
lonely = pd.DataFrame({"runs_10": [100], "wickets_10": [4]})

# GHALAT: akele match pe naya scaler fit karna
wrong_scaler = StandardScaler()
wrong_scaler.fit(lonely)

print("Mean seekha:", wrong_scaler.mean_)   # [100, 4], yaani khud wahi match
wrong_scaler.transform(lonely)               # (0, 0)

Mean seekha: [100.   4.]


array([[0., 0.]])

In [9]:
# SAHI: training data se seekha hua scaler
scaler.transform(lonely)                     # (2, 2)

array([[2., 2.]])

### Natija
Ek match ka average **khud wahi match** hai, to `(100 - 100) = 0`. Har naya match **"average"** ban jata, aur model har dafa ek hi jawab deta.
Sahi scaler ne training ka mean (80, 2) aur std (10, 1) lagaya aur sahi jawab **(2, 2)** diya.

### Experiment 2: Train + test mila ke fit karna
`small` = training (4 matches), `new_matches` = test (2 matches).

In [10]:
all_data = pd.concat([small, new_matches])

# GHALAT: train aur test mila ke fit
mixed_scaler = StandardScaler()
mixed_scaler.fit(all_data)

print("Sirf train se mean:  ", scaler.mean_)        # [80, 2]
print("Train + test se mean:", mixed_scaler.mean_)  # [83.33, 2.33]

Sirf train se mean:   [80.  2.]
Train + test se mean: [83.33333333  2.33333333]


### Natija
Mean 80 se **83.33** ho gaya: **test matches ki information** training ki scaling mein ghus gayi. Ye **data leakage** hai.

Test data **future** ki naqal hai. Model bante waqt future ke matches hote hi nahi, to unki information istemal karna dhoka hai.
Model test pe achha lagega, lekin asal duniya mein waisa nahi chalega.

---
## Khulasa: Rule

```
fit        ->  SIRF training data (ek dafa: mean aur std seekho)
transform  ->  training data, test data, aur har naya match
               (hamesha wahi seekha hua mean aur std)
```

Yahi rule **OneHotEncoder** pe bhi lagta hai (Lesson 4.3).